# Transformer + frozen GloVe

Runs the whole experiment in two rounds, dev only (test is never touched here):

1. **Round 1:** the default and the 7 ablations in `src/sc4002/experiments.py`, seed 42 each. Each
   ablation changes one setting from the default.
2. **Round 2:** the 3 best round-1 ablations and the default, with seeds 42, 43 and 44. The final
   configuration is the one with the highest **mean** dev combined score.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/SC4002_WL'
RES = f'{WORK}/results'
os.makedirs(f'{WORK}/embeddings', exist_ok=True)
os.makedirs(RES, exist_ok=True)
os.environ['SC4002_EMB_DIR'] = f'{WORK}/embeddings'
os.environ['SC4002_RESULTS'] = RES
os.environ['SC4002_CKPT'] = f'{WORK}/checkpoints'

## Get the code
**Option A** (after WL_branch is pushed): clone it.
**Option B**: upload `wl_code.zip` (made locally) to `MyDrive/SC4002_WL/` and unzip it.

In [ ]:
CODE = '/content/SC4002'
USE_GIT = False 

if USE_GIT:
    !rm -rf {CODE} && git clone -q -b WL_branch https://github.com/masamune-prog/SC4002.git {CODE}
else:
    !rm -rf {CODE} && mkdir -p {CODE} && unzip -q -o {WORK}/wl_code.zip -d {CODE}
!pip install -q tokenizers pytest
os.environ['PYTHONPATH'] = f'{CODE}/src'
import sys; sys.path.insert(0, f'{CODE}/src')
%cd {CODE}

## GloVe 6B word vectors (downloaded once to Drive, ~860 MB)

In [ ]:
E = os.environ['SC4002_EMB_DIR']
if not os.path.exists(f'{E}/glove.6B.zip'):
    !wget -q -O {E}/glove.6B.zip https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
!ls -la {E}

## Unit tests

In [ ]:
!python -m pytest -q tests

## Round 1: default + 7 ablations, seed 42

In [ ]:
from sc4002.experiments import CORE
!python -m sc4002.train --exp {' '.join(CORE)} --seeds 42

In [ ]:
import json, pandas as pd

def load_runs():
    rows = [json.loads(l) for l in open(f'{RES}/ablations.jsonl') if l.strip()]
    return pd.DataFrame([{'exp': r['exp'], 'seed': r['config']['seed'], 'best_epoch': r['best_epoch'],
                          'intent': r['dev_intent_acc'], 'slot': r['dev_slot_f1'], 'combined': r['dev_combined'],
                          'total_params': r['total_params'], 'trainable_params': r['trainable_params'],
                          'minutes': r['train_time'] / 60} for r in rows])

r1 = load_runs().query('seed == 42').set_index('exp')
missing = [e for e in CORE if e not in r1.index]
assert not missing, f'round 1 not finished, missing: {missing}'
r1['delta_vs_default'] = r1['combined'] - r1.loc['default', 'combined']
r1.loc[CORE].sort_values('combined', ascending=False).round(2)

## Round 2: top-3 ablations and the default, seeds 42, 43, 44

In [ ]:
TOP3 = r1.drop(index='default').sort_values('combined', ascending=False).index[:3].tolist()
CANDIDATES = ['default'] + TOP3
SEEDS = [42, 43, 44]
print('Round 2 candidates:', CANDIDATES)
!python -m sc4002.train --exp {' '.join(CANDIDATES)} --seeds {' '.join(map(str, SEEDS))}

## 3-seed comparison and final choice
Selection rule: highest mean dev combined = (intent acc + slot F1) / 2 over the three seeds.

In [ ]:
df = load_runs()
df = df[df.exp.isin(CANDIDATES) & df.seed.isin(SEEDS)]
missing = [(e, s) for e in CANDIDATES for s in SEEDS if not ((df.exp == e) & (df.seed == s)).any()]
assert not missing, f'runs still missing: {missing}'
print(df.sort_values(['exp', 'seed']).round(2).to_string(index=False))
summary = df.groupby('exp').agg(intent=('intent', 'mean'), slot=('slot', 'mean'),
                                combined=('combined', 'mean'), combined_std=('combined', 'std'),
                                min_combined=('combined', 'min'), total_params=('total_params', 'first'),
                                trainable_params=('trainable_params', 'first'))
summary['delta_vs_default'] = summary['combined'] - summary.loc['default', 'combined']
summary = summary.sort_values('combined', ascending=False)
summary.round(2)

In [ ]:
from sc4002.experiments import EXPERIMENTS
best = summary.index[0]
d = summary.loc[best, 'delta_vs_default']
noise = summary['combined_std'].max()
print(f'Final configuration: {best}  (mean dev combined {summary.loc[best, "combined"]:.2f}, '
      f'{d:+.2f} vs default; largest seed std {noise:.2f})')
if best != 'default' and d < noise:
    print('Note: the margin over the default is smaller than the seed spread, so the difference is not clearly real.')
print('Overrides vs default:', EXPERIMENTS[best])